In [ ]:
# Note, the checkpointer keeps track of the context, therefore to be able to reject or edit, the context has to be clean and not include a response stating that the email was already sent!
# For example, this would happen if you tried to run the reject code right after the approve!

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending - this is where the real sending logic would exist, like calling the email API, etc
    return f"Email sent"

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    system_prompt="Use the tools provided to interact respond to email requests.",
    model="ollama:gemma4",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        # Generally, there are 3 main reasons for including human in the loop middleware: sensitive actions, handling missing context, debugging agents
        # This example will go over a sensitive action, however, the same approach would be taken for handling missing context and debugging agents!
        # docs: https://docs.langchain.com/oss/python/langchain/human-in-the-loop (see configuration options section for config specifics)
        HumanInTheLoopMiddleware(
            interrupt_on={ # specifies which tools do and do not require approval
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [5]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    { # I tweaked the messages and email values, as the agent did not understand the prompts the same way as in the langchain module video.
        "messages": [HumanMessage(content="Please read the email from john@example.com in the state and send a response immediately. Write and send the reply now in the same thread.")],
        "email": "From john@example.com. Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [6]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'I '
                                                                          'understand '
                                                                          'things '
                                                                          'come '
                                                                          'up. '
                                                                          'Can '
               

In [28]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': "Hi John, Thanks for letting me know. Yes, let's definitely reschedule. Could you send over a few times/days that work best for you next week, or feel free to send a calendar invite? I'll look forward to connecting then. Best, [Your Name]"}, 'description': 'Tool execution requires approval\n\nTool: send_email\nArgs: {\'body\': "Hi John, Thanks for letting me know. Yes, let\'s definitely reschedule. Could you send over a few times/days that work best for you next week, or feel free to send a calendar invite? I\'ll look forward to connecting then. Best, [Your Name]"}'}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='49cfd5815f71ee1354f400a0c8cb9178')]


In [7]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, No problem at all. I understand things come up. Can you let me know a few times that work better for you next week? We can find something that suits both of us. Best, [My Name]


## Approve

In [29]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "From john@example.com. Hi Seán, I'm going to be late for our "
          'meeting tomorrow. Can we reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='857418a2-cc5f-4c65-8140-a15915dbe997'),
              AIMessage(content='I need an email address to read your email. Could you please provide the address you would like me to check?', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-28T21:42:11.995782Z', 'done': True, 'done_reason': 'stop', 'total_duration': 17627421833, 'load_duration': 3491833, 'prompt_eval_count': 126, 'prompt_eval_duration': 399020000, 'eval_count': 505, 'eval_duration': 17210668000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a18476-663e-7481-ab17-f12fa23cfd1b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens'

## Reject

In [9]:
from langgraph.types import Command
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off the email with the following: Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "From john@example.com. Hi Seán, I'm going to be late for our "
          'meeting tomorrow. Can we reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read the email from john@example.com in the state and send a response immediately. Write and send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='aaba1316-f689-47c9-af85-205a6447ecd4'),
              HumanMessage(content='Please read the email from john@example.com in the state and send a response immediately. Write and send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='5854b94d-e843-449a-a6b4-ae8991f494e4'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-11-13T12:22:18.852173Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12625243542, 'load_duration': 3183625, 'prompt_eval_count': 172, 'prompt_eval_duration': 171504000, 'eval_count': 345, 'eval_duration': 12

In [10]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Edit

In [37]:
# The rejecting and approving process can be cumbersome, especially if the agent comes back with a response that you don't quite like!
# The "edit" type may be the right choice in these instances
# When you use "edit", the agent will not ask you to approve, the "edit" comes with a built-in approval
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "From john@example.com. Hi Seán, I'm going to be late for our "
          'meeting tomorrow. Can we reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='857418a2-cc5f-4c65-8140-a15915dbe997'),
              AIMessage(content='I need an email address to read your email. Could you please provide the address you would like me to check?', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-28T21:42:11.995782Z', 'done': True, 'done_reason': 'stop', 'total_duration': 17627421833, 'load_duration': 3491833, 'prompt_eval_count': 126, 'prompt_eval_duration': 399020000, 'eval_count': 505, 'eval_duration': 17210668000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a18476-663e-7481-ab17-f12fa23cfd1b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens'